In [ ]:
# @title 📦 Setup (run this cell first)
%matplotlib inline
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from IPython.display import display, HTML, clear_output

# Matplotlib style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Okabe-Ito palette
COLORS = {
    'text': '#E69F00', 'image': '#0072B2', 'connection': '#56B4E9',
    'failure': '#D55E00', 'crossmodal': '#CC79A7', 'neutral': '#999999'
}

print('✅ Setup complete.')

In [ ]:
# @title 🔧 Helper Functions (collapsed)

SHAPES = ['circle', 'square', 'triangle']
COLORS_MAP = {
    'red': (230, 25, 25),
    'blue': (25, 25, 230),
    'green': (25, 204, 25),
}
SIZES = {'small': 5, 'large': 10}
POSITIONS = {
    'center': (16, 16),
    'offset': [(8, 8), (8, 24), (24, 8), (24, 24)],
}

def generate_shape(shape, color, size='large', position='center', img_size=32):
    img = Image.new('RGB', (img_size, img_size), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    rgb = COLORS_MAP[color]
    radius = SIZES[size]
    if position == 'center':
        cx, cy = 16, 16
    else:
        rng = np.random.RandomState()
        cx, cy = POSITIONS['offset'][rng.randint(len(POSITIONS['offset']))]
    if shape == 'circle':
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'square':
        draw.rectangle([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'triangle':
        pts = [(cx, cy-radius), (cx-radius, cy+radius), (cx+radius, cy+radius)]
        draw.polygon(pts, fill=rgb)
    return np.array(img, dtype=np.float32) / 255.0

def generate_dataset(include_size=False, include_position=False):
    sizes = list(SIZES.keys()) if include_size else ['large']
    positions = ['center', 'offset'] if include_position else ['center']
    dataset = []
    for color in COLORS_MAP:
        for shape in SHAPES:
            for sz in sizes:
                for pos in positions:
                    img = generate_shape(shape, color, size=sz, position=pos)
                    label = f"{sz} {color} {shape}" if include_size else f"{color} {shape}"
                    dataset.append((img, label))
    return dataset

class ShapeDataset(torch.utils.data.Dataset):
    def __init__(self, include_size=False, include_position=False):
        raw = generate_dataset(include_size=include_size, include_position=include_position)
        self.images, self.labels, self.label_texts = [], [], []
        unique_labels = []
        for _, label in raw:
            if label not in unique_labels:
                unique_labels.append(label)
        self.label_to_idx = {l: i for i, l in enumerate(unique_labels)}
        for img, label in raw:
            tensor = torch.from_numpy(img).permute(2, 0, 1)
            self.images.append(tensor)
            self.labels.append(self.label_to_idx[label])
            self.label_texts.append(label)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.label_texts[idx]

# --- Visualization helpers ---
def plot_image_grid(images, titles=None, ncols=3, figsize=None, suptitle=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None:
        figsize = (ncols * 2.5, nrows * 2.5)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                if img.dim() == 3 and img.shape[0] in (1, 3):
                    img = img.permute(1, 2, 0)
                img = img.detach().cpu().numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=12)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_denoising_filmstrip(images, steps=None, title='Denoising Process'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, torch.Tensor):
            if img.dim() == 3 and img.shape[0] in (1, 3):
                img = img.permute(1, 2, 0)
            img = img.detach().cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        label = f't={steps[i]}' if steps else f'Step {i}'
        ax.set_title(label, fontsize=10)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_attention_heatmap(attn_weights, text_tokens, grid_size=8, title='Attention Weights'):
    if isinstance(attn_weights, torch.Tensor):
        attn_weights = attn_weights.detach().cpu().numpy()
    if attn_weights.ndim == 3:
        attn_weights = attn_weights[0]
    n_tokens = len(text_tokens)
    fig, axes = plt.subplots(1, n_tokens, figsize=(3 * n_tokens, 3))
    if n_tokens == 1:
        axes = [axes]
    for i, (ax, token) in enumerate(zip(axes, text_tokens)):
        heatmap = attn_weights[:, i].reshape(grid_size, grid_size)
        im = ax.imshow(heatmap, cmap='Blues', vmin=0)
        ax.set_title(f'"{token}"', fontsize=12)
        ax.axis('off')
        for r in range(grid_size):
            for c in range(grid_size):
                val = heatmap[r, c]
                if val > 0.1:
                    ax.text(c, r, f'{val:.1f}', ha='center', va='center',
                            fontsize=7, color='white' if val > 0.5 else 'black')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_guidance_comparison(images, scales, title='Guidance Scale Comparison'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 3))
    if n == 1:
        axes = [axes]
    labels_map = {1.0: 'Weak', 3.0: 'Moderate', 7.5: 'Balanced', 10.0: 'Strong', 15.0: 'Extreme'}
    for ax, img, scale in zip(axes, images, scales):
        if isinstance(img, torch.Tensor):
            if img.dim() == 3 and img.shape[0] in (1, 3):
                img = img.permute(1, 2, 0)
            img = img.detach().cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        label = labels_map.get(scale, '')
        ax.set_title(f'{label}\n(scale={scale})', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

print('✅ Helpers loaded.')

In [ ]:
# @title 🏗️ Models & Pre-trained Weights (collapsed)

VOCAB = [
    'red', 'blue', 'green', 'circle', 'square', 'triangle',
    'small', 'large', 'center', 'offset', '<pad>', '<unk>'
]
WORD_TO_IDX = {w: i for i, w in enumerate(VOCAB)}

def tokenize(text, max_len=4):
    tokens = []
    for word in text.lower().split():
        tokens.append(WORD_TO_IDX.get(word, WORD_TO_IDX['<unk>']))
    if len(tokens) < max_len:
        tokens += [WORD_TO_IDX['<pad>']] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return torch.tensor(tokens, dtype=torch.long)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=12, embed_dim=32, hidden_dim=64, out_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=WORD_TO_IDX['<pad>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, token_ids):
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class ImageEncoder(nn.Module):
    def __init__(self, input_dim=3072, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)
    def forward(self, images):
        x = images.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class MiniCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()
    def forward(self, images, token_ids):
        return self.text_encoder(token_ids), self.image_encoder(images)
    def compute_loss(self, text_emb, image_emb, tau=0.07):
        logits = torch.matmul(text_emb, image_emb.t()) / tau
        labels = torch.arange(len(logits), device=logits.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2

class CrossAttention(nn.Module):
    def __init__(self, img_dim=128, text_dim=32, head_dim=32):
        super().__init__()
        self.W_q = nn.Linear(img_dim, head_dim)
        self.W_k = nn.Linear(text_dim, head_dim)
        self.W_v = nn.Linear(text_dim, head_dim)
        self.W_out = nn.Linear(head_dim, img_dim)
        self.scale = head_dim ** -0.5
    def forward(self, image_features, text_features):
        Q = self.W_q(image_features)
        K = self.W_k(text_features)
        V = self.W_v(text_features)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn, dim=-1)
        out = torch.matmul(attn_weights, V)
        out = self.W_out(out)
        return out, attn_weights

class MicroUNet(nn.Module):
    def __init__(self, time_emb_dim=32, text_emb_dim=32):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(time_emb_dim, 128), nn.ReLU())
        self.cross_attn = CrossAttention(img_dim=128, text_dim=text_emb_dim, head_dim=32)
        self.dec1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.final = nn.Conv2d(64, 3, 3, padding=1)
    def forward(self, x, t_emb, text_emb):
        out, _ = self.forward_with_attention(x, t_emb, text_emb)
        return out
    def forward_with_attention(self, x, t_emb, text_emb):
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        t = self.time_mlp(t_emb)
        e3 = e3 + t[:, :, None, None]
        B, C, H, W = e3.shape
        patches = e3.permute(0, 2, 3, 1).reshape(B, H * W, C)
        text_feat = text_emb.unsqueeze(1)
        attn_out, attn_weights = self.cross_attn(patches, text_feat)
        e3 = attn_out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        d1 = F.relu(self.dec1(e3))
        d1 = torch.cat([d1, e2], dim=1)
        d2 = F.relu(self.dec2(d1))
        d2 = torch.cat([d2, e1], dim=1)
        out = self.final(d2)
        return out, attn_weights

class NoiseScheduler:
    def __init__(self, T=50, s=0.008):
        self.T = T
        self.s = s
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
        alpha_bars = f / f[0]
        alpha_bars = alpha_bars.clamp(min=1e-5, max=1.0)
        betas = 1 - alpha_bars[1:] / alpha_bars[:-1]
        betas = betas.clamp(min=0.0001, max=0.999)
        self.betas = betas.float()
        self.alphas = (1 - self.betas).float()
        self.alpha_bars = torch.cumprod(self.alphas, dim=0).float()
    def add_noise(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        alpha_bar_t = self.alpha_bars[t]
        while alpha_bar_t.dim() < x_0.dim():
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        return torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
    def sample_step(self, x_t, t, predicted_noise):
        beta_t = self.betas[t]
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bars[t]
        while beta_t.dim() < x_t.dim():
            beta_t = beta_t.unsqueeze(-1)
            alpha_t = alpha_t.unsqueeze(-1)
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        mean = (1 / torch.sqrt(alpha_t)) * (x_t - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise)
        if isinstance(t, int) and t == 0:
            return mean
        if isinstance(t, torch.Tensor) and (t == 0).all():
            return mean
        noise = torch.randn_like(x_t)
        return mean + torch.sqrt(beta_t) * noise
    @staticmethod
    def get_time_embedding(t, dim=32):
        if isinstance(t, int):
            t = torch.tensor([t], dtype=torch.float32)
        elif isinstance(t, torch.Tensor) and t.dim() == 0:
            t = t.unsqueeze(0).float()
        else:
            t = t.float()
        half_dim = dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
        emb = t.unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

@torch.no_grad()
def generate_image(model, scheduler, text_emb, null_text_emb=None,
                   guidance_scale=7.5, steps=50, seed=42):
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    intermediates = []
    save_steps = set([steps - 1] + [int(i * (steps - 1) / 7) for i in range(8)])
    for t_idx in reversed(range(steps)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        if null_text_emb is not None and guidance_scale != 1.0:
            noise_cond = model(x, t_emb, text_emb)
            noise_uncond = model(x, t_emb, null_text_emb)
            predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            predicted_noise = model(x, t_emb, text_emb)
        x = scheduler.sample_step(x, t_idx, predicted_noise)
        if t_idx in save_steps:
            intermediates.append(x[0].clone().clamp(0, 1))
    intermediates.reverse()
    return x[0].clamp(0, 1), intermediates

@torch.no_grad()
def generate_with_attention(model, scheduler, text_emb, null_text_emb=None,
                            guidance_scale=7.5, steps=50, seed=42, attn_step=25):
    """Generate and capture attention weights at a specific step."""
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    captured_attn = None
    for t_idx in reversed(range(steps)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        if t_idx == attn_step:
            out, attn_w = model.forward_with_attention(x, t_emb, text_emb)
            captured_attn = attn_w
            noise_cond = out
        else:
            noise_cond = model(x, t_emb, text_emb)
        if null_text_emb is not None and guidance_scale != 1.0:
            noise_uncond = model(x, t_emb, null_text_emb)
            predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            predicted_noise = noise_cond
        x = scheduler.sample_step(x, t_idx, predicted_noise)
    return x[0].clamp(0, 1), captured_attn

def train_clip(model, dataset, epochs=100, lr=1e-3, tau=0.07):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    for epoch in range(epochs):
        optimizer.zero_grad()
        text_emb, image_emb = model(images, token_ids)
        loss = model.compute_loss(text_emb, image_emb, tau=tau)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

def train_denoiser(model, dataset, scheduler, text_encoder, epochs=300, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    with torch.no_grad():
        text_embs = text_encoder(token_ids)
    for epoch in range(epochs):
        optimizer.zero_grad()
        B = len(images)
        t = torch.randint(0, scheduler.T, (B,))
        noise = torch.randn_like(images)
        x_t = scheduler.add_noise(images, t, noise)
        t_emb = scheduler.get_time_embedding(t)
        predicted_noise = model(x_t, t_emb, text_embs)
        loss = F.mse_loss(predicted_noise, noise)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 50 == 0:
            print(f'Denoiser Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    return losses


def color_accuracy(image, expected_color):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    mask = brightness > 0.1
    if mask.sum() == 0:
        return False
    avg_color = image[mask].mean(axis=0)
    channel_map = {'red': 0, 'blue': 2, 'green': 1}
    return np.argmax(avg_color) == channel_map[expected_color]

def shape_accuracy(image, expected_shape):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    gen_mask = (brightness > 0.15).astype(np.float32)
    if gen_mask.sum() < 5:
        return False
    templates = {}
    for shape in ['circle', 'square', 'triangle']:
        tpl = Image.new('L', (32, 32), 0)
        draw = ImageDraw.Draw(tpl)
        r, cx, cy = 10, 16, 16
        if shape == 'circle':
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'square':
            draw.rectangle([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'triangle':
            draw.polygon([(cx, cy-r), (cx-r, cy+r), (cx+r, cy+r)], fill=255)
        templates[shape] = np.array(tpl, dtype=np.float32) / 255.0
    gen_norm = gen_mask - gen_mask.mean()
    gen_std = gen_norm.std()
    if gen_std < 1e-6:
        return False
    best_shape, best_corr = None, -2.0
    for shape, tpl_mask in templates.items():
        tpl_norm = tpl_mask - tpl_mask.mean()
        tpl_std = tpl_norm.std()
        if tpl_std < 1e-6:
            continue
        corr = (gen_norm * tpl_norm).sum() / (gen_std * tpl_std * gen_mask.size)
        if corr > best_corr:
            best_corr = corr
            best_shape = shape
    return best_shape == expected_shape

# --- Load all models ---
import os
dataset = ShapeDataset()
scheduler = NoiseScheduler(T=50)
clip_model = MiniCLIP()
unet = MicroUNet()

weights_loaded = False
for path in ['weights/full_pipeline.pt', '../weights/full_pipeline.pt',
             '/content/weights/full_pipeline.pt']:
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        clip_model.text_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_text'].items()})
        clip_model.image_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_image'].items()})
        unet.load_state_dict(
            {k: v.float() for k, v in checkpoint['unet'].items()})
        weights_loaded = True
        print(f'✅ Loaded pre-trained weights from {path}')
        break

if not weights_loaded:
    print('⚠️ Pre-trained weights not found. Training from scratch (~4 min)...')
    train_clip(clip_model, dataset, epochs=100)
    clip_model.eval()
    train_denoiser(unet, dataset, scheduler, clip_model.text_encoder, epochs=300)
    print('✅ Training complete.')

clip_model.eval()
unet.eval()

# Pre-compute null embedding for CFG
null_tokens = tokenize('<pad> <pad>')
null_text_emb = clip_model.text_encoder(null_tokens.unsqueeze(0))

# Pre-compute text embeddings for all 9 prompts
all_prompts = [dataset[i][2] for i in range(len(dataset))]
all_text_embs = {}
for prompt in all_prompts:
    tokens = tokenize(prompt)
    all_text_embs[prompt] = clip_model.text_encoder(tokens.unsqueeze(0))

print('✅ All models ready.')

In [ ]:
# Runtime check
try:
    _ = len(COLORS)
    _ = unet
    print('✅ Ready to go!')
except NameError:
    print('⚠️ Please run the setup cells above (grey cells 1-3).')
    print('   Click the ▶ button on each grey cell at the top.')

# 💬 Notebook 4: The Conversation — Cross-Attention

You have a text encoder and an image generator. **They’re strangers.** Cross-attention introduces them.

In the last notebook, the UNet learned to remove noise. But how does the text prompt actually *steer* what the model generates? The answer is **cross-attention** — a mechanism where each part of the image checks the text at every denoising step.

In [ ]:
# DEMO: Cross-attention ON vs OFF
# Same model, same noise, same seed. Only difference: does the model listen to text?

prompt = 'red circle'
text_emb = all_text_embs[prompt]
zero_emb = torch.zeros_like(text_emb)  # No text signal

# Generate WITH text guidance
img_with, frames_with = generate_image(unet, scheduler, text_emb, null_text_emb,
                                        guidance_scale=7.5, seed=42)
# Generate WITHOUT text (zero embedding, no guidance)
img_without, frames_without = generate_image(unet, scheduler, zero_emb, None,
                                              guidance_scale=1.0, seed=42)

fig = plot_denoising_filmstrip(frames_with,
    title=f'✅ Cross-attention ON: "{prompt}"')
plt.show()

fig = plot_denoising_filmstrip(frames_without,
    title='❌ Cross-attention OFF: no text signal')
plt.show()

print('Same model. Same starting noise. Same seed.')
print('The ONLY difference: cross-attention connects (or disconnects) text and image.')
print('Without text guidance, the model produces an incoherent blob.')

---

> **🧭 Orient:** *"Cross-attention is the driver asking the co-pilot at every single turn."*
>
> At each of the 50 denoising steps, the model doesn’t just remove noise blindly. It checks the text: *"What should I be making here?"* Cross-attention is that conversation.

---

### How Cross-Attention Works: Q / K / V

Think of it as a conversation between the image and the text:

| Role | Source | What it does |
|------|--------|--------------|
| **Q** (Query) | 🖼️ Image patches | Each image region *asks*: "What should I generate?" |
| **K** (Key) | 📝 Text tokens | Text *offers* topics to match against |
| **V** (Value) | 📝 Text tokens | Text *provides* the actual information |

**The flow:**
1. Each image patch creates a **question** (Q)
2. Each text token creates a **topic label** (K) and an **answer** (V)
3. Questions match against topics → attention weights
4. High-attention answers guide what each patch generates

> Q asks, K matches, V answers.

In [ ]:
# CORE INTERACTIVE: Attention Heatmap Explorer
# See WHERE the model looks at text for each image region.

# Pre-compute attention maps and generated images for all 9 prompts
print('Pre-computing attention maps for all 9 prompts...')
attn_cache = {}
img_cache = {}
for prompt in all_prompts:
    text_emb = all_text_embs[prompt]
    img, attn = generate_with_attention(unet, scheduler, text_emb, null_text_emb,
                                         guidance_scale=7.5, seed=42, attn_step=25)
    attn_cache[prompt] = attn
    img_cache[prompt] = img
print('✅ Done!')

@interact(prompt=Dropdown(options=all_prompts, value=all_prompts[0], description='Prompt:'))
def explore_attention(prompt):
    attn = attn_cache[prompt]  # (1, 64, 1) since single text embedding
    gen_img = img_cache[prompt]

    attn_np = attn[0].detach().cpu().numpy()  # (64, 1)
    heatmap = attn_np[:, 0].reshape(8, 8)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Generated image
    img_show = gen_img.permute(1, 2, 0).numpy()
    axes[0].imshow(np.clip(img_show, 0, 1))
    axes[0].set_title(f'Generated: "{prompt}"', fontsize=13)
    # Add 8x8 grid overlay
    for i in range(9):
        axes[0].axhline(y=i*4 - 0.5, color='white', linewidth=0.5, alpha=0.3)
        axes[0].axvline(x=i*4 - 0.5, color='white', linewidth=0.5, alpha=0.3)
    axes[0].axis('off')

    # Attention heatmap
    im = axes[1].imshow(heatmap, cmap='Blues', vmin=0)
    axes[1].set_title(f'Attention: "{prompt}"', fontsize=13)
    # Numeric values
    for r in range(8):
        for c in range(8):
            val = heatmap[r, c]
            color = 'white' if val > heatmap.max() * 0.6 else 'black'
            axes[1].text(c, r, f'{val:.2f}', ha='center', va='center',
                        fontsize=7, color=color)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], shrink=0.8)

    # Overlay: attention on image
    # Upscale heatmap to 32x32
    heatmap_up = np.repeat(np.repeat(heatmap, 4, axis=0), 4, axis=1)
    axes[2].imshow(np.clip(img_show, 0, 1))
    axes[2].imshow(heatmap_up, cmap='Blues', alpha=0.5, vmin=0)
    axes[2].set_title('Overlay: Where attention is highest', fontsize=13)
    axes[2].axis('off')

    fig.tight_layout()
    plt.show()

    # Analysis
    max_idx = np.unravel_index(heatmap.argmax(), heatmap.shape)
    print(f'Highest attention at patch ({max_idx[0]}, {max_idx[1]}) = {heatmap.max():.3f}')
    center_attn = heatmap[2:6, 2:6].mean()
    edge_attn = (heatmap.sum() - heatmap[2:6, 2:6].sum()) / (64 - 16)
    print(f'Center region average: {center_attn:.3f}')
    print(f'Edge region average:   {edge_attn:.3f}')

---

> **Insight:** Notice how the attention pattern changes with different prompts. Color tokens tend to attend **broadly** (color is a global property), while shape information concentrates on the **center patches** (where the shape lives). The model discovered these patterns from data alone.

---

In [ ]:
# BUILD: The CrossAttention module
# This is the core mechanism that connects text to image.

class CrossAttentionDemo(nn.Module):
    """Q=image asks, K=text matches, V=text answers."""
    def __init__(self, img_dim=128, text_dim=32, head_dim=32):
        super().__init__()
        self.W_q = nn.Linear(img_dim, head_dim)    # Image -> Question
        self.W_k = nn.Linear(text_dim, head_dim)    # Text -> Key
        self.W_v = nn.Linear(text_dim, head_dim)    # Text -> Value
        self.W_out = nn.Linear(head_dim, img_dim)   # Back to image dim
        self.scale = head_dim ** -0.5

    def forward(self, image_features, text_features):
        Q = self.W_q(image_features)                # Each patch asks
        K = self.W_k(text_features)                 # Text offers topics
        V = self.W_v(text_features)                 # Text offers answers
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn, dim=-1)      # Who matches whom?
        out = self.W_out(attn_weights @ V)          # Weighted answers
        return out, attn_weights

print('CrossAttention: 4 linear layers (W_q, W_k, W_v, W_out) + softmax')
demo_ca = CrossAttentionDemo()
print(f'Parameters: {sum(p.numel() for p in demo_ca.parameters()):,}')
print('\nW_out is CRITICAL: it projects the attention output back to image dimensions.')
print('Without W_out, the dimensions would not match and the model would fail.')

In [ ]:
# STUMBLE: What happens with UNIFORM attention? (Co-pilot with earplugs)

# Save original weights
original_attn_state = {k: v.clone() for k, v in unet.cross_attn.state_dict().items()}

# Monkey-patch: force uniform attention weights
original_forward = unet.cross_attn.forward
def uniform_forward(image_features, text_features):
    Q = unet.cross_attn.W_q(image_features)
    K = unet.cross_attn.W_k(text_features)
    V = unet.cross_attn.W_v(text_features)
    attn = torch.matmul(Q, K.transpose(-2, -1)) * unet.cross_attn.scale
    # FORCE UNIFORM: every patch attends equally
    attn_weights = torch.ones_like(attn) / attn.shape[-1]
    out = torch.matmul(attn_weights, V)
    out = unet.cross_attn.W_out(out)
    return out, attn_weights

unet.cross_attn.forward = uniform_forward

# Generate with uniform attention
prompt_stumble = 'red circle'
text_emb_stumble = all_text_embs[prompt_stumble]
img_uniform, _ = generate_image(unet, scheduler, text_emb_stumble, null_text_emb,
                                 guidance_scale=7.5, seed=42)

# Restore original
unet.cross_attn.forward = original_forward

# Compare
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

img_uniform_np = img_uniform.permute(1, 2, 0).numpy()
axes[0].imshow(np.clip(img_uniform_np, 0, 1))
axes[0].set_title('❌ Uniform attention\n(co-pilot with earplugs)', fontsize=12,
                   color=COLORS['failure'])
axes[0].axis('off')

img_normal = img_cache[prompt_stumble].permute(1, 2, 0).numpy()
axes[1].imshow(np.clip(img_normal, 0, 1))
axes[1].set_title(f'✅ Learned attention\n("{prompt_stumble}")', fontsize=12,
                   color=COLORS['connection'])
axes[1].axis('off')

fig.suptitle('What happens when every patch attends equally?', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print('With uniform attention, every image patch gets the same "average" text signal.')
print('Result: a blurry, incoherent blob. No spatial differentiation.')

---

> **💡 Aha!** Uniform attention is like a co-pilot with earplugs. The driver hears a muffled average of all directions at once. **Selective listening creates structure.** The model needs to attend *differently* to different parts of the text for different image regions.

---

In [ ]:
# FIX: Learned attention restored! Compare the attention maps.

# Uniform attention map (all equal)
uniform_map = np.ones((8, 8)) / 64

# Learned attention map
learned_attn = attn_cache['red circle'][0].detach().cpu().numpy()[:, 0].reshape(8, 8)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

im1 = axes[0].imshow(uniform_map, cmap='Blues', vmin=0, vmax=learned_attn.max())
axes[0].set_title('❌ Uniform attention', fontsize=12, color=COLORS['failure'])
for r in range(8):
    for c in range(8):
        axes[0].text(c, r, f'{uniform_map[r,c]:.2f}', ha='center', va='center',
                    fontsize=7, color='black')
axes[0].axis('off')

im2 = axes[1].imshow(learned_attn, cmap='Blues', vmin=0)
axes[1].set_title('✅ Learned attention', fontsize=12, color=COLORS['connection'])
for r in range(8):
    for c in range(8):
        val = learned_attn[r, c]
        color = 'white' if val > learned_attn.max() * 0.6 else 'black'
        axes[1].text(c, r, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=color)
axes[1].axis('off')

fig.suptitle('Uniform vs Learned Attention Maps ("red circle")',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print('Learned attention concentrates where it matters most.')
print('This selective focus is what enables the model to generate coherent shapes.')

In [ ]:
# Guidance Scale Explorer
# How loudly should the co-pilot speak?

# Pre-compute images at 5 guidance scales
guidance_scales = [1.0, 3.0, 7.5, 10.0, 15.0]
guidance_prompt = 'red circle'
text_emb_guide = all_text_embs[guidance_prompt]

print('Pre-computing images at different guidance scales...')
guidance_images = []
for gs in guidance_scales:
    img, _ = generate_image(unet, scheduler, text_emb_guide, null_text_emb,
                             guidance_scale=gs, seed=42)
    guidance_images.append(img)
print('✅ Done!')

@interact(scale=Dropdown(
    options=['All scales', '1.0 (Weak)', '3.0 (Moderate)', '7.5 (Balanced)',
             '10.0 (Strong)', '15.0 (Extreme)'],
    value='All scales', description='Scale:'))
def explore_guidance(scale):
    if scale == 'All scales':
        fig = plot_guidance_comparison(guidance_images, guidance_scales,
                                       title=f'Guidance Scale for "{guidance_prompt}"')
        plt.show()
        print('Low scale (1.0) = co-pilot whispers. Model is creative but unfocused.')
        print('High scale (15.0) = co-pilot shouts. Model follows closely but may over-saturate.')
        print('Sweet spot is usually 7-10.')
    else:
        gs_val = float(scale.split('(')[0].strip())
        idx = guidance_scales.index(gs_val)
        fig, ax = plt.subplots(figsize=(4, 4))
        img = guidance_images[idx].permute(1, 2, 0).numpy()
        ax.imshow(np.clip(img, 0, 1))
        labels_map = {1.0: 'Weak', 3.0: 'Moderate', 7.5: 'Balanced',
                      10.0: 'Strong', 15.0: 'Extreme'}
        ax.set_title(f'Scale={gs_val} ({labels_map[gs_val]})', fontsize=14)
        ax.axis('off')
        plt.show()

In [ ]:
# Compositional Challenge: Can the model handle different combinations?

challenge_prompts = ['red circle', 'blue square', 'red square',
                      'green triangle', 'blue circle', 'green square']
challenge_images = []
challenge_results = []

for prompt in challenge_prompts:
    text_emb = all_text_embs.get(prompt)
    if text_emb is None:
        tokens = tokenize(prompt)
        text_emb = clip_model.text_encoder(tokens.unsqueeze(0))
    img, _ = generate_image(unet, scheduler, text_emb, null_text_emb,
                             guidance_scale=7.5, seed=42)
    challenge_images.append(img)
    parts = prompt.split()
    c_ok = color_accuracy(img, parts[0])
    s_ok = shape_accuracy(img, parts[1])
    challenge_results.append((c_ok, s_ok))

# Display
fig, axes = plt.subplots(2, 3, figsize=(9, 7))
axes = axes.flatten()
for i, (ax, img, prompt) in enumerate(zip(axes, challenge_images, challenge_prompts)):
    im = img.permute(1, 2, 0).numpy()
    ax.imshow(np.clip(im, 0, 1))
    c_ok, s_ok = challenge_results[i]
    c_sym = '✅' if c_ok else '❌'
    s_sym = '✅' if s_ok else '❌'
    ax.set_title(f'"{prompt}"\nColor {c_sym}  Shape {s_sym}', fontsize=11)
    ax.axis('off')

fig.suptitle('Compositional Challenge: Mixing Colors and Shapes',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

successes = sum(1 for c, s in challenge_results if c and s)
print(f'\nFull success: {successes}/{len(challenge_prompts)}')
print('\nWhen the model gets it wrong, it\'s often an "attribute binding" problem:')
print('the model knows BOTH the color and shape exist, but attaches them incorrectly.')
print('This is the same problem that makes "a red cat on a blue chair" hard for DALL-E.')


---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Manager’s Briefing: Compositional Accuracy

Cross-attention enables text to guide image generation, but it has a fundamental limitation: **attribute binding**. When the prompt contains multiple objects with different properties (e.g., "a red cat on a blue chair"), the model may swap attributes between objects.

**Why this matters:** If your use case requires precise multi-attribute generation (product mockups, design prototypes), you’ll encounter this limitation at scale.

**Questions to ask your vendor:**
1. *"How does your model handle multi-object prompts?"*
2. *"What’s your compositional accuracy benchmark score?"*
3. *"Do you support region-specific prompting (e.g., inpainting)?"*

</div>

---

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Real-World Story: The E-Commerce Material Swap

An e-commerce company used AI to generate product images from descriptions. The prompt "silver watch on a wooden table" sometimes produced a wooden-looking watch on a metallic table. The model swapped the material attributes between objects.

**Formal diagnosis:** Cross-attention has no hard constraint that forces attributes to bind to specific objects. The attention mechanism can “bleed” — material tokens attend to the wrong spatial regions. This is an active research problem (layout-guided generation, attention manipulation).

**As a decision-maker:** If attribute precision is critical, consider models with layout control or generate objects separately and composite them.

</div>

---

---

### 📋 Co-Pilot Reference Card (rows 1–4)

| Notebook | Component | What It Does | Co-Pilot Metaphor |
|----------|-----------|-------------|-------------------|
| NB1 | Raw Ingredients | Images = tensors of numbers; text = embedding vectors; cosine sim measures closeness | "Here are the parts of the car and the map" |
| NB2 | CLIP (Translator) | Learns shared embedding space; contrastive loss pulls matching pairs together | "The co-pilot learns to read the map" |
| NB3 | Diffusion (Engine) | Forward: add noise. Reverse: predict & remove noise. UNet preserves spatial structure. | "The driver learns to navigate by being dropped at random locations" |
| **NB4** | **Cross-Attention (Conversation)** | **Q=image, K/V=text. Each image region checks the text. Guidance scale = co-pilot volume.** | **"At every turn, the driver asks the co-pilot"** |

---